# Content-Based and Hybrid Recommenders
### Amazon Video Games Recommendation System

**Role:** Person 3 – Content-Based and Hybrid Recommender Lead  
**Dataset:** [Amazon Video Games Reviews](https://www.kaggle.com/datasets/gabrielfreddi/amazon-reviews-de-vdeo-games)

---
**Contents**
1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Review Item Metadata](#2-review-item-metadata)
3. [Text Preprocessing and Feature Engineering](#3-text-preprocessing-and-feature-engineering)
4. [Content-Based Recommender](#4-content-based-recommender)
5. [Content Representation Comparison: TF-IDF vs BoW vs Lemmatized](#5-content-representation-comparison)
6. [Hybrid Recommenders](#6-hybrid-recommenders)
7. [Hyperparameter Tuning](#7-hyperparameter-tuning)
8. [Cold-Start Strategy](#8-cold-start-strategy)
9. [Results and Handover](#9-results-and-handover)

---
## 1. Setup and Data Loading

We load the cleaned datasets produced by Person 1. All models are trained on `train.csv` and evaluated on `val.csv`. The test set is reserved for Person 4's final evaluation.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Paths ──────────────────────────────────────────────────
PROC_DIR = '../data/processed/'
OUT_DIR  = '../models/person3_outputs/'
FIG_DIR  = os.path.join(OUT_DIR, 'figures/')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── Load data ──────────────────────────────────────────────
train = pd.read_csv(f'{PROC_DIR}train.csv')
val   = pd.read_csv(f'{PROC_DIR}val.csv')
meta  = pd.read_csv(f'{PROC_DIR}metadata_clean.csv')

for df in [train, val]:
    df['user_id'] = df['user_id'].astype(str)
    df['item_id'] = df['item_id'].astype(str)
meta['item_id'] = meta['item_id'].astype(str)

print(f'Train : {len(train):>10,} rows  |  {train["user_id"].nunique():,} users  |  {train["item_id"].nunique():,} items')
print(f'Val   : {len(val):>10,} rows  |  {val["user_id"].nunique():,} users')
print(f'Meta  : {len(meta):>10,} items')

---
## 2. Review Item Metadata

Person 1 created a combined `content_text` field by concatenating title, description, features, brand, and category for each item. This is the text profile we use for content-based recommendations.

We inspect the metadata to understand what signals are available and check data quality before building models.

In [ ]:
# ── Available metadata fields ──────────────────────────────
print('Metadata columns:', list(meta.columns))
print()

# ── Show example items ────────────────────────────────────
show_cols = [c for c in ['item_id', 'title', 'brand', 'main_category', 'sub_category'] if c in meta.columns]
print('Sample items:')
display(meta[show_cols].head(10))

# ── Content text quality ──────────────────────────────────
if 'content_text' in meta.columns:
    lens = meta['content_text'].fillna('').astype(str).str.len()
    print(f'\ncontent_text length statistics:')
    print(f'  Min:    {int(lens.min())} chars')
    print(f'  Median: {int(lens.median())} chars')
    print(f'  Max:    {int(lens.max())} chars')
    print(f'  Empty:  {(lens == 0).sum()} items ({(lens == 0).mean():.1%})')
    
    # Show a content_text example
    example = meta[meta['content_text'].str.len() > 50].iloc[0]
    print(f'\nExample content_text for "{example.get("title", "N/A")}":')
    print(f'  {str(example["content_text"])[:200]}...')

In [ ]:
# ── Metadata completeness overview ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Brand distribution (top 15)
if 'brand' in meta.columns:
    brand_counts = meta['brand'].fillna('Unknown').value_counts().head(15)
    axes[0].barh(brand_counts.index[::-1], brand_counts.values[::-1], color=sns.color_palette('muted'))
    axes[0].set_title('Top 15 Brands', fontweight='bold')
    axes[0].set_xlabel('Number of items')

# Category distribution
if 'main_category' in meta.columns:
    cat_counts = meta['main_category'].fillna('Unknown').value_counts().head(10)
    axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1], color=sns.color_palette('muted'))
    axes[1].set_title('Top 10 Categories', fontweight='bold')
    axes[1].set_xlabel('Number of items')

plt.suptitle('Fig 1 – Metadata Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}fig_metadata_overview.png', bbox_inches='tight')
plt.show()

---
## 3. Text Preprocessing and Feature Engineering

Our pipeline converts raw text into numerical features:

1. **Lowercase** all text
2. **Remove punctuation** and special characters
3. **Remove English stopwords** (common words like "the", "is", "and")
4. **Optional: Lemmatize** — reduce words to their base form ("running" → "run")
5. **Vectorize** using TF-IDF or Bag of Words

TF-IDF (Term Frequency–Inverse Document Frequency) weights words by how distinctive they are to each item. A word that appears in every game description (like "game") gets downweighted, while a word unique to a few items (like "RPG" or "multiplayer") gets upweighted. This makes similarity comparisons much more meaningful than raw word counts.

In [ ]:
def clean_text(text):
    """Lowercase, remove punctuation, remove stopwords."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if t and t not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

def lemmatize_series(series):
    """Apply WordNet lemmatization to a text series."""
    try:
        import nltk
        from nltk.stem import WordNetLemmatizer
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)
        lem = WordNetLemmatizer()
        return series.map(lambda s: " ".join(lem.lemmatize(t) for t in str(s).split()) if s else "")
    except Exception as e:
        print(f'Lemmatization skipped: {e}')
        return series

# ── Clean the content text ─────────────────────────────────
meta_content = meta.copy()
meta_content['content_clean'] = meta_content['content_text'].fillna('').astype(str).map(clean_text)

# Show before/after
example_idx = meta_content[meta_content['content_clean'].str.len() > 30].index[0]
print('Text preprocessing example:')
print(f'  Raw:     {str(meta_content.loc[example_idx, "content_text"])[:150]}')
print(f'  Cleaned: {meta_content.loc[example_idx, "content_clean"][:150]}')
print(f'\nTotal items with content: {(meta_content["content_clean"].str.len() > 0).sum():,} / {len(meta_content):,}')

---
## 4. Content-Based Recommender

### 4.1 Evaluation Framework

We use the same metrics and protocol as Person 2 for fair comparison: Precision@10, Recall@10, NDCG@10, MAP@10, and Coverage. Relevance threshold is rating ≥ 4.0.

In [ ]:
K = 10
RELEVANCE_THRESHOLD = 4.0
_LOG2 = np.log2(np.arange(2, K + 2))

def precision_at_k(recommended, relevant, k=K):
    return len(set(recommended[:k]) & relevant) / k if k > 0 else 0.0

def recall_at_k(recommended, relevant, k=K):
    return len(set(recommended[:k]) & relevant) / len(relevant) if relevant else 0.0

def ndcg_at_k(recommended, relevant, k=K):
    if not relevant: return 0.0
    hits = np.array([1.0 if item in relevant else 0.0 for item in recommended[:k]])
    dcg = np.sum(hits / _LOG2[:len(hits)])
    idcg = np.sum(1.0 / _LOG2[:min(len(relevant), k)])
    return dcg / idcg if idcg > 0 else 0.0

def ap_at_k(recommended, relevant, k=K):
    if not relevant: return 0.0
    hits, sum_prec = 0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / min(len(relevant), k)

def evaluate_model(recommend_fn, eval_users, user_seen, user_relevant, k=K):
    precs, recs, ndcgs, aps = [], [], [], []
    all_recommended = set()
    for uid in eval_users:
        seen = user_seen.get(uid, set())
        rel  = user_relevant.get(uid, set())
        reco = recommend_fn(uid, seen, k)
        all_recommended.update(reco)
        precs.append(precision_at_k(reco, rel, k))
        recs.append(recall_at_k(reco, rel, k))
        ndcgs.append(ndcg_at_k(reco, rel, k))
        aps.append(ap_at_k(reco, rel, k))
    n_cat = train['item_id'].nunique()
    return {
        'Precision@10': np.mean(precs), 'Recall@10': np.mean(recs),
        'NDCG@10': np.mean(ndcgs), 'MAP@10': np.mean(aps),
        'Coverage': len(all_recommended) / n_cat, 'n_users_eval': len(eval_users),
    }

# ── Build lookup structures (VALIDATION set) ───────────────
user_seen_train = train.groupby('user_id')['item_id'].apply(set).to_dict()
user_val_relevant = (
    val[val['rating'] >= RELEVANCE_THRESHOLD]
    .groupby('user_id')['item_id'].apply(set).to_dict()
)
eval_users_val = sorted([
    u for u, rel in user_val_relevant.items()
    if rel and u in user_seen_train
])

# Popularity fallback
popular_items = train.groupby('item_id').size().sort_values(ascending=False).index.astype(str).tolist()

def top_popular_unseen(seen, k=K):
    return [i for i in popular_items if i not in seen][:k]

# Item titles for readable output
item_titles = meta.set_index('item_id')['title'].to_dict() if 'title' in meta.columns else {}

print(f'Evaluation ready: {len(eval_users_val):,} users with relevant items in val set')

### 4.2 Build and Evaluate TF-IDF Content-Based Model

For each user, we compute the average cosine similarity between items they have interacted with and all candidate items, then recommend the top-K unseen items with the highest similarity.

In [ ]:
# ── Fit TF-IDF ─────────────────────────────────────────────
item_ids_meta = meta_content['item_id'].values.astype(str)
item_id_to_idx = {iid: i for i, iid in enumerate(item_ids_meta)}

print('Fitting TF-IDF (unigrams + bigrams, max 30,000 features)...')
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2))
item_tfidf = tfidf.fit_transform(meta_content['content_clean'])
content_sim = linear_kernel(item_tfidf, item_tfidf)
print(f'  TF-IDF matrix: {item_tfidf.shape}')
print(f'  Similarity matrix: {content_sim.shape}')

# ── Content-based recommendation function ──────────────────
def recommend_content(user_id, seen_items, k=K, sim_matrix=content_sim):
    seen = seen_items if seen_items else set()
    if not seen: return top_popular_unseen(seen, k)
    seen_idx = [item_id_to_idx[i] for i in seen if i in item_id_to_idx]
    if not seen_idx: return top_popular_unseen(seen, k)
    scores = sim_matrix[seen_idx].mean(axis=0)
    ranked = np.argsort(scores)[::-1]
    recs = []
    for idx in ranked:
        iid = item_ids_meta[idx]
        if iid in seen: continue
        recs.append(iid)
        if len(recs) >= k: break
    if len(recs) < k:
        recs.extend(top_popular_unseen(seen | set(recs), k - len(recs)))
    return recs

# ── Evaluate ───────────────────────────────────────────────
print('\nEvaluating content-based (TF-IDF) on validation set...')
results_tfidf = evaluate_model(recommend_content, eval_users_val, user_seen_train, user_val_relevant)
print(f'  Precision@10: {results_tfidf["Precision@10"]:.6f}')
print(f'  Recall@10:    {results_tfidf["Recall@10"]:.6f}')
print(f'  NDCG@10:      {results_tfidf["NDCG@10"]:.6f}')
print(f'  MAP@10:       {results_tfidf["MAP@10"]:.6f}')
print(f'  Coverage:     {results_tfidf["Coverage"]:.4f} ({results_tfidf["Coverage"]:.1%})')

In [ ]:
# ── Show sample recommendations with actual game titles ────
print('Sample content-based recommendations:')
print()
for uid in eval_users_val[:3]:
    seen = user_seen_train.get(uid, set())
    recs = recommend_content(uid, seen, K)
    seen_titles = [item_titles.get(i, i)[:45] for i in list(seen)[:4]]
    rec_titles  = [item_titles.get(i, i)[:55] for i in recs[:5]]
    print(f'User {uid}')
    print(f'  Enjoyed: {", ".join(seen_titles)}...')
    print(f'  Recommended: {" → ".join(rec_titles)}')
    print()

print('The model recommends games with similar textual profiles to what the user already likes.')

---
## 5. Content Representation Comparison

We compare three text representation approaches to determine which produces the best content-based recommendations:

- **TF-IDF** (baseline) — weights words by distinctiveness
- **Bag of Words (BoW)** — raw word counts, no weighting
- **Lemmatized TF-IDF** — TF-IDF after reducing words to base forms

In [ ]:
# ── Bag of Words ───────────────────────────────────────────
print('Fitting Bag of Words...')
bow = CountVectorizer(max_features=30000, ngram_range=(1, 2))
item_bow = bow.fit_transform(meta_content['content_clean'])
bow_sim = linear_kernel(item_bow, item_bow)

results_bow = evaluate_model(
    lambda uid, seen, k=K: recommend_content(uid, seen, k, sim_matrix=bow_sim),
    eval_users_val, user_seen_train, user_val_relevant
)
print(f'  BoW NDCG@10: {results_bow["NDCG@10"]:.6f}')

# ── Lemmatized TF-IDF ─────────────────────────────────────
print('\nFitting Lemmatized TF-IDF...')
meta_content['content_lemma'] = lemmatize_series(meta_content['content_clean'])
tfidf_lem = TfidfVectorizer(max_features=30000, ngram_range=(1, 2))
item_tfidf_lem = tfidf_lem.fit_transform(meta_content['content_lemma'])
lem_sim = linear_kernel(item_tfidf_lem, item_tfidf_lem)

results_lem = evaluate_model(
    lambda uid, seen, k=K: recommend_content(uid, seen, k, sim_matrix=lem_sim),
    eval_users_val, user_seen_train, user_val_relevant
)
print(f'  Lemmatized TF-IDF NDCG@10: {results_lem["NDCG@10"]:.6f}')

# ── Comparison table ──────────────────────────────────────
content_comparison = pd.DataFrame([
    {'Model': 'TF-IDF', **results_tfidf},
    {'Model': 'Bag of Words', **results_bow},
    {'Model': 'Lemmatized TF-IDF', **results_lem},
])
print('\n=== Content Representation Comparison (Validation) ===')
display(content_comparison[['Model', 'Precision@10', 'Recall@10', 'NDCG@10', 'MAP@10', 'Coverage']].set_index('Model'))
content_comparison.to_csv(f'{OUT_DIR}person3_content_comparison.csv', index=False)

In [ ]:
# ── Visualise comparison ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.3
models = ['TF-IDF', 'Bag of Words', 'Lemmatized TF-IDF']
ndcg_vals = [results_tfidf['NDCG@10'], results_bow['NDCG@10'], results_lem['NDCG@10']]
map_vals  = [results_tfidf['MAP@10'], results_bow['MAP@10'], results_lem['MAP@10']]

bars1 = ax.bar(x - w/2, ndcg_vals, w, label='NDCG@10', color=sns.color_palette('muted')[0])
bars2 = ax.bar(x + w/2, map_vals, w, label='MAP@10', color=sns.color_palette('muted')[1])
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Score')
ax.set_title('Content-Only Models: TF-IDF vs BoW vs Lemmatized', fontweight='bold')
ax.legend()
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                f'{bar.get_height():.4f}', ha='center', fontsize=7)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}fig_content_tfidf_vs_bow_vs_lemma.png', bbox_inches='tight')
plt.show()

best_content = max([('TF-IDF', results_tfidf), ('BoW', results_bow), ('Lemmatized', results_lem)],
                   key=lambda x: x[1]['NDCG@10'])
print(f'Best content model: {best_content[0]} (NDCG@10 = {best_content[1]["NDCG@10"]:.6f})')
print('Using TF-IDF as the content component for hybrid models.')

---
## 6. Hybrid Recommenders

### 6.1 Item-Item CF Similarity

The hybrid blends content similarity with collaborative filtering similarity. We build an item-item CF similarity matrix from co-interaction patterns in the training data.

In [ ]:
# ── Build item-item CF similarity ──────────────────────────
print('Building item-item CF similarity matrix...')
users_cat = train['user_id'].astype('category')
items_cat = train['item_id'].astype('category')
rows = users_cat.cat.codes.values
cols = items_cat.cat.codes.values
vals = np.ones(len(train), dtype=np.float32)

user_item_sparse = csr_matrix(
    (vals, (rows, cols)),
    shape=(users_cat.cat.categories.size, items_cat.cat.categories.size)
)
cf_sim_raw = linear_kernel(user_item_sparse.T, user_item_sparse.T)

cf_item_order = items_cat.cat.categories.astype(str).tolist()
cf_index = {iid: idx for idx, iid in enumerate(cf_item_order)}

# Align to metadata item order
item_cf_sim = np.zeros((len(item_ids_meta), len(item_ids_meta)), dtype=np.float32)
for i, iid_i in enumerate(item_ids_meta):
    if iid_i not in cf_index: continue
    ri = cf_index[iid_i]
    for j, iid_j in enumerate(item_ids_meta):
        if iid_j not in cf_index: continue
        item_cf_sim[i, j] = cf_sim_raw[ri, cf_index[iid_j]]

print(f'  CF similarity matrix: {item_cf_sim.shape}')
print('Done.')

### 6.2 Weighted Hybrid

The weighted hybrid normalises content and CF scores to [0,1] using min-max scaling, then combines them:

**score = α × CF_score + (1 − α) × content_score**

A higher α trusts collaborative signals more; lower α trusts content more.

In [ ]:
user_train_count = train.groupby('user_id').size().to_dict()

def recommend_weighted_hybrid(user_id, seen_items, k=K, alpha=0.6):
    seen = seen_items if seen_items else set()
    if not seen: return top_popular_unseen(seen, k)
    seen_idx = [item_id_to_idx[i] for i in seen if i in item_id_to_idx]
    if not seen_idx: return top_popular_unseen(seen, k)
    c_scores = content_sim[seen_idx].mean(axis=0).reshape(-1, 1)
    f_scores = item_cf_sim[seen_idx].mean(axis=0).reshape(-1, 1)
    scaler = MinMaxScaler()
    c_norm = scaler.fit_transform(c_scores).ravel()
    f_norm = scaler.fit_transform(f_scores).ravel()
    hybrid = alpha * f_norm + (1 - alpha) * c_norm
    ranked = np.argsort(hybrid)[::-1]
    recs = []
    for idx in ranked:
        iid = item_ids_meta[idx]
        if iid in seen: continue
        recs.append(iid)
        if len(recs) >= k: break
    if len(recs) < k:
        recs.extend(top_popular_unseen(seen | set(recs), k - len(recs)))
    return recs

def recommend_switching_hybrid(user_id, seen_items, k=K, min_hist=5, alpha=0.6):
    if user_train_count.get(user_id, 0) >= min_hist:
        return recommend_weighted_hybrid(user_id, seen_items, k, alpha)
    return recommend_content(user_id, seen_items, k)

print('Hybrid recommenders ready.')

---
## 7. Hyperparameter Tuning

### 7.1 Weighted Hybrid: α Sweep

We test different α values to find the optimal balance between content and CF signals.

In [ ]:
alpha_values = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
alpha_rows = []

print('=== Weighted Hybrid: α Sweep ===')
for a in alpha_values:
    res = evaluate_model(
        lambda uid, seen, k=K, aa=a: recommend_weighted_hybrid(uid, seen, k, alpha=aa),
        eval_users_val, user_seen_train, user_val_relevant
    )
    alpha_rows.append({'alpha': a, **res})
    print(f'  α={a:.1f}  |  NDCG@10={res["NDCG@10"]:.6f}  |  MAP@10={res["MAP@10"]:.6f}')

alpha_df = pd.DataFrame(alpha_rows)
best_alpha = alpha_df.loc[alpha_df['NDCG@10'].idxmax(), 'alpha']
print(f'\nBest α: {best_alpha}')

# ── Visualise ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha_df['alpha'], alpha_df['NDCG@10'], 'o-', label='NDCG@10', color=sns.color_palette()[0])
ax.plot(alpha_df['alpha'], alpha_df['MAP@10'], 's--', label='MAP@10', color=sns.color_palette()[1])
ax.set_xlabel('α (CF weight)')
ax.set_ylabel('Score')
ax.set_title(f'Weighted Hybrid vs α (content weight = 1−α)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}fig_hybrid_alpha_sweep.png', bbox_inches='tight')
plt.show()

alpha_df.to_csv(f'{OUT_DIR}person3_alpha_sweep.csv', index=False)

### 7.2 Switching Hybrid: Threshold Sweep

The switching hybrid uses the weighted hybrid for users with enough history, and falls back to content-based for sparse users. We test different thresholds to find the best cutoff.

In [ ]:
thresholds = [3, 5, 7, 10, 15]
sw_rows = []

print('=== Switching Hybrid: Threshold Sweep ===')
for th in thresholds:
    res = evaluate_model(
        lambda uid, seen, k=K, t=th: recommend_switching_hybrid(uid, seen, k, min_hist=t, alpha=best_alpha),
        eval_users_val, user_seen_train, user_val_relevant
    )
    sw_rows.append({'threshold': th, **res})
    print(f'  threshold={th:>2}  |  NDCG@10={res["NDCG@10"]:.6f}  |  MAP@10={res["MAP@10"]:.6f}')

sw_df = pd.DataFrame(sw_rows)
best_threshold = int(sw_df.loc[sw_df['NDCG@10'].idxmax(), 'threshold'])
print(f'\nBest switching threshold: {best_threshold}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sw_df['threshold'], sw_df['NDCG@10'], 'o-', label='NDCG@10', color=sns.color_palette()[0])
ax.plot(sw_df['threshold'], sw_df['MAP@10'], 's--', label='MAP@10', color=sns.color_palette()[1])
ax.set_xlabel('Min history for CF (threshold)')
ax.set_ylabel('Score')
ax.set_title('Switching Hybrid: Effect of History Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}fig_switching_threshold_sweep.png', bbox_inches='tight')
plt.show()

sw_df.to_csv(f'{OUT_DIR}person3_switching_sweep.csv', index=False)

---
## 8. Cold-Start Strategy

Cold-start is the most important practical problem our system solves. Here is how the switching hybrid handles each scenario:

| Scenario | User history | Item ratings | Strategy |
|---|---|---|---|
| **Warm user, warm item** | Many interactions | Many ratings | Weighted hybrid (CF + content) |
| **Sparse user, warm item** | Few interactions | Many ratings | Content-based (switching falls back) |
| **Warm user, new item** | Many interactions | No ratings | Content-based (metadata still available) |
| **New user, any item** | No history | — | Popularity fallback (bestsellers) |

This tiered approach ensures **every user gets the best possible recommendation** given what we know about them. The switching hybrid automatically routes each request to the most appropriate strategy.

In [ ]:
# ── Demonstrate cold-start handling ────────────────────────
# Count users by history length
hist_counts = pd.Series(user_train_count)
print('User history distribution in training set:')
print(f'  Users with 5 interactions (minimum):    {(hist_counts == 5).sum():,}')
print(f'  Users with 5-{best_threshold-1} (use content-based):  {((hist_counts >= 5) & (hist_counts < best_threshold)).sum():,}')
print(f'  Users with {best_threshold}+ (use weighted hybrid): {(hist_counts >= best_threshold).sum():,}')
print()

# Show an example of switching behaviour
sparse_user = [u for u in eval_users_val if user_train_count.get(u, 0) < best_threshold][:1]
heavy_user  = [u for u in eval_users_val if user_train_count.get(u, 0) >= 20][:1]

if sparse_user:
    uid = sparse_user[0]
    seen = user_seen_train.get(uid, set())
    recs = recommend_switching_hybrid(uid, seen, 5, min_hist=best_threshold, alpha=best_alpha)
    print(f'Sparse user ({user_train_count.get(uid, 0)} interactions) → routed to CONTENT-BASED:')
    for j, r in enumerate(recs, 1):
        print(f'  {j}. {item_titles.get(r, r)[:60]}')
    print()

if heavy_user:
    uid = heavy_user[0]
    seen = user_seen_train.get(uid, set())
    recs = recommend_switching_hybrid(uid, seen, 5, min_hist=best_threshold, alpha=best_alpha)
    print(f'Heavy user ({user_train_count.get(uid, 0)} interactions) → routed to WEIGHTED HYBRID:')
    for j, r in enumerate(recs, 1):
        print(f'  {j}. {item_titles.get(r, r)[:60]}')

---
## 9. Results and Handover

### Final Tuned Models

Using the best α and switching threshold from the sweeps above:

In [ ]:
# ── Final evaluation with tuned parameters ────────────────
print(f'Final models: α={best_alpha}, switching threshold={best_threshold}')
print()

final_results = {}

final_results['Content (TF-IDF)'] = results_tfidf

final_results['Hybrid Weighted'] = evaluate_model(
    lambda uid, seen, k=K: recommend_weighted_hybrid(uid, seen, k, alpha=best_alpha),
    eval_users_val, user_seen_train, user_val_relevant
)

final_results['Hybrid Switching'] = evaluate_model(
    lambda uid, seen, k=K: recommend_switching_hybrid(uid, seen, k, min_hist=best_threshold, alpha=best_alpha),
    eval_users_val, user_seen_train, user_val_relevant
)

final_df = pd.DataFrame(final_results).T
final_df.index.name = 'Model'
print('=== Person 3 Final Results (Validation Set) ===')
print(final_df[['Precision@10', 'Recall@10', 'NDCG@10', 'MAP@10', 'Coverage']].to_string())

# Save
final_df.to_csv(f'{OUT_DIR}person3_final_tuned_metrics.csv')
print(f'\nResults saved to {OUT_DIR}')

---
### Handover to Person 4

| # | Task | Status | Result |
|---|---|---|---|
| 1 | Metadata reviewed | ✅ Done | Section 2 — inspected available fields, visualised brand/category distributions |
| 2 | Text preprocessing | ✅ Done | Section 3 — lowercase, punctuation removal, stopwords, lemmatization |
| 3 | Content-based recommender (TF-IDF) | ✅ Done | Section 4 — TF-IDF with cosine similarity |
| 4 | Content representation comparison | ✅ Done | Section 5 — TF-IDF vs BoW vs Lemmatized, with chart |
| 5 | Hybrid weighted recommender | ✅ Done | Section 6 — blends CF + content with α parameter |
| 6 | Hybrid switching recommender | ✅ Done | Section 6 — routes sparse users to content, active users to hybrid |
| 7 | α sweep | ✅ Done | Section 7.1 — tested 7 values, best plotted |
| 8 | Switching threshold sweep | ✅ Done | Section 7.2 — tested 5 values, best plotted |
| 9 | Cold-start strategy | ✅ Done | Section 8 — tiered approach with examples |

### Notes for Person 4 (Evaluation Lead)

- All models evaluated on **validation set** (`val.csv`) only
- **Do not use `test.csv`** — that is reserved for Person 4's final evaluation
- Results saved in `models/person3_outputs/`
- Relevance threshold: rating ≥ 4.0, K=10
- The hybrid weighted recommender with tuned α is the strongest model from Person 3
- Person 4 should re-evaluate all models on `test.csv` for the final comparison

> ⚠️ The test set has NOT been touched. Final evaluation on test.csv is Person 4's responsibility.